<a href="https://colab.research.google.com/github/dohyung-kim/ccri/blob/main/script/adm0/pop_exposure_gee_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee
import geemap
import pandas as pd
import os
import json
import geemap
from pathlib import Path
# Initialize
path_to_ee_auth = '../.secrets/ee_auth.json'
key_path = Path(path_to_ee_auth)
key_file = key_path.read_text()
key_dict = json.loads(key_file)
email = key_dict["client_email"]

auth = ee.ServiceAccountCredentials(email=email, key_data=key_file)
ee.Initialize(auth)

In [2]:
# Define admin level
admin_level = 'adm0'

# Set output folder
output_folder = f'p1_exposure_{admin_level}'


In [8]:
# Load child population and replace one image
org_childpop = ee.ImageCollection("projects/unicef-ccri/assets/childpop_constrained")
image_to_replace_id = 'tha_T_Under_18_2024_CN_100m_R2024A_v1'
new_image = ee.Image(f'projects/unicef-ccri/assets/{image_to_replace_id}').set('system:index', image_to_replace_id)
filtered_collection = org_childpop.filter(ee.Filter.neq('system:index', image_to_replace_id))
childpop = filtered_collection.merge(ee.ImageCollection([new_image])).mosaic()
scale = filtered_collection.first().projection().nominalScale()
print(f'scale: {scale.getInfo()}')
totalpop = ee.Image("projects/unicef-ccri/assets/worldpop_1km")
totalpop_res = totalpop.projection().nominalScale()

reference_image = ee.Image("projects/unicef-ccri/assets/heatwave_frequency_2014_2023_avg")
target_scale = reference_image.projection().nominalScale()
target_crs = reference_image.projection().crs()

# Hazard list
hazards = [
    # {"id": "projects/unicef-ccri/assets/river_flood_r100", "threshold": 0.01, "name": "river_flood_100yr_jrc_2024"},
    {"id": "projects/unicef-ccri/assets/coastal_flood_r100", "threshold": 0, "name": "coastal_flood_100yr_jrc_2024"},
    # {"id": "projects/unicef-ccri/assets/storm_giri_rp100", "threshold": 17.5, "name": "tropical_storm_100yr_giri_2024"},
    # {"id": "projects/unicef-ccri/assets/ASI_return_level_100yr", "threshold": 30, "name": "agricultural_drought_fao_1984-2023"},
    # {"id": "projects/unicef-ccri/assets/spei12_period_mean_2014_2024", "threshold": -1, "name": "drought_spei_copernicus_1940-2024"},
    # {"id": "projects/unicef-ccri/assets/spi12_period_mean_2014_2024", "threshold": -1, "name": "drought_spi_copernicus_1940-2024"},
    # {"id": "projects/unicef-ccri/assets/heatwave_frequency_return_level_100yr", "threshold": 'Mean', "name": "heatwave_frequency_ecmwf_2014-2024"},
    # {"id": "projects/unicef-ccri/assets/heatwave_duration_return_level_100yr", "threshold": 'Mean', "name": "heatwave_duration_ecmwf_2014-2024"},
    # {"id": "projects/unicef-ccri/assets/heatwave_severity_return_level_100yr", "threshold": 'Mean', "name": "heatwave_severity_ecmwf_2014-2024"},
    # {"id": "projects/unicef-ccri/assets/high_temp_degree_days_return_level_100yr", "threshold": 35, "name": "extreme_heat_ecmwf_2014-2024"},
    # {"id": "projects/unicef-ccri/assets/FIRMS_FRP_90th_percentile", "threshold": 'Mean', "name": "fire_FRP_nasa_2001-2024"},
    # {"id": "projects/unicef-ccri/assets/FIRMS_count_90th_percentile", "threshold": 'Mean', "name": "fire_frequency_nasa_2001-2023"},
    # {"id": "projects/unicef-ccri/assets/sand_dust_storm_annual", "threshold": 0, "name": "sand_dust_storm_unccd_2024"},
    # {"id": "projects/unicef-ccri/assets/pm25_p90_1998_2023", "threshold": 5, "name": "air_pollution_pm25_1998-2023"},
    # {"id": "projects/unicef-ccri/assets/Pv_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapv_2012-2022"},
    # {"id": "projects/unicef-ccri/assets/Pf_average_2013_2022", "threshold": 0.001, "name": "vectorborne_malariapf_2012-2022"}
]

# Load country boundaries
admin_fc_path = f'projects/unicef-ccri/assets/{admin_level}_wfp'
country_fc = ee.FeatureCollection(admin_fc_path)
simple_fc = ee.FeatureCollection(f'projects/unicef-ccri/assets/{admin_level}_simple')
adm_ids = country_fc.aggregate_array(f'{admin_level}_id').distinct().getInfo()
global_geom = simple_fc.geometry()

# Update thresholds if needed
for hazard in hazards:
    if hazard['threshold'] == 'Mean':
        layer = (
            ee.ImageCollection(hazard['id']).mosaic()
            if hazard['name'] in ["river_flood_100yr_jrc_2024", "coastal_flood_100yr_jrc_2024", "tropical_storm_100yr_giri_2024"]
            else ee.Image(hazard['id'])
        )
        th = layer.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=global_geom,
            scale=target_scale,
            bestEffort=True,
            maxPixels=1e13
        ).values().get(0)
        if th is not None:
            hazard['threshold'] = ee.Number(th).getInfo()
        else:
            print(f"⚠️ Skipping hazard {hazard['name']} — global mean threshold is null.")
            hazard['skip'] = True


th_shape_area = country_fc.filter(ee.Filter.eq(f'{admin_level}_id', 122)).first().getNumber('Shape_Area')

# Loop through admin regions
iso3s = ["COL"]
for iso3 in iso3s:
    print(iso3)
    if iso3 is None:
        continue

    # Filter feature by adm_id
    # print(country_fc.getInfo())
    feature = country_fc.filter(ee.Filter.eq('iso3', "COL")).first()
    # Get stscod

    stscod = feature.get('stscod').getInfo()

    # Skip if stscod is not 'State'
    if stscod != 'State':
        print(f"Skipping adm_id {iso3} due to stscod = {stscod}")
        continue

    if feature.get('iso3').getInfo() != 'COL':
        print(f"Skipping adm_id {iso3} due to iso3 = {feature.get('iso3').getInfo()}")
        continue

    print(f"Submitting task for {admin_level}_id: {iso3}...")

    region = country_fc.filter(ee.Filter.eq('iso3', "COL")).first()
    shape_area = region.getNumber('Shape_Area')

    # Choose simplification level based on area
    simplification_tolerance = ee.Algorithms.If(
        shape_area.gt(th_shape_area),
        10000,
        100
    )

    simplified_geom = region.geometry().simplify(simplification_tolerance)
    region_geom = ee.Geometry(simplified_geom)

    iso3 = region.get('iso3')
    region_name = region.get(f'{admin_level}_name')

    childpop_sum = childpop.reduceRegion(
        ee.Reducer.sum(), region_geom, scale=scale, crs='EPSG:4326', maxPixels=1e13).get('b1')
    totalpop_sum = totalpop.reduceRegion(
        ee.Reducer.sum(), region_geom, scale=totalpop_res, crs='EPSG:4326', maxPixels=1e13).get('b1')

    features = []

    for hazard in hazards:
        if hazard.get('skip'):
            continue
        name = hazard['name']
        TH = hazard['threshold']

        layer = (
            ee.ImageCollection(hazard['id']).mosaic()
            if hazard['name'] in ["river_flood_100yr_jrc_2024", "coastal_flood_100yr_jrc_2024", "tropical_storm_100yr_giri_2024"]
            else ee.Image(hazard['id'])
        )

        if name == "agricultural_drought_fao_1984-2023":
            layer = layer.updateMask(layer.lte(100))
            exposed = childpop.updateMask(layer.gt(ee.Image.constant(TH)))
        elif name in ["drought_spei_copernicus_1940-2024", "drought_spi_copernicus_1940-2024"]:
            exposed = childpop.updateMask(layer.lt(ee.Image.constant(TH)))
        else:
            exposed = childpop.updateMask(layer.gt(ee.Image.constant(TH)))

        exposed_sum = exposed.reduceRegion(
            ee.Reducer.sum(), region_geom, scale=scale, crs='EPSG:4326', maxPixels=1e13).get('b1')

        print(f"exposed_sum: {exposed_sum.getInfo()}")

    #     feature = ee.Feature(None, {
    #         'iso3': iso3,
    #         f'{admin_level}_name': region_name,
    #         f'{admin_level}_id': adm_id,
    #         'hazard': name,
    #         'child_population_exposed': exposed_sum,
    #         'child_population_total': childpop_sum,
    #         'population_total': totalpop_sum
    #     })
    #     features.append(feature)

    # print(ee.FeatureCollection(features).getInfo())
    print(th_shape_area.getInfo())
    print(simplification_tolerance.getInfo())
    break

print("✅ All tasks submitted.")

scale: 92.76624195666344
COL
Submitting task for adm0_id: COL...
exposed_sum: 29475.016295027777
33.147591296895925
10000
✅ All tasks submitted.


In [ ]:
import os
import glob
import pandas as pd

# Define the folder and get all CSV file paths
hazard_folder = os.path.join('/content/drive/MyDrive', output_folder)
hazard_files = glob.glob(f'{hazard_folder}/*.csv')

# Read and concatenate all CSVs into one DataFrame
merged_df = pd.concat([pd.read_csv(f) for f in hazard_files], ignore_index=True)

# Output directory
output_dir = '/content/drive/MyDrive/p1_exposure'
os.makedirs(output_dir, exist_ok=True)

# Loop through each unique hazard and export a CSV
for hazard_name in merged_df['hazard'].unique():
    # Filter the data
    df_subset = merged_df[merged_df['hazard'] == hazard_name]

    # Define full path
    output_path = os.path.join(output_dir, f"{hazard_name}_exposure_adm0.csv")

    # Save the CSV
    df_subset.to_csv(output_path, index=False)

    print(f"✅ Saved: {output_path}")
